# 3 · Geometry Metadata

**Author:** Héctor Fernández Pinacho  
**Lab:** IDEAL Lab · ETH Zürich

Derives all geometry-based metadata from the LHS sample and the generated STL files. The measured masses of the bench-tested propellers calibrate the effective SLS print density, every STL is converted into mass and spin-axis inertia at that density, and a single validation-based linear correction (`izz_regressed = a · izz + b`) aligns the STL-predicted inertia with the trifilar-pendulum measurements. The per-station NACA airfoil codes that the XFoil stage needs are generated from the sampled airfoil parameters. Finally, the same mass/inertia and NACA computations are run for the 10 recovered validation propellers (whose true printed geometry differs from the main geometry table, see `utils/measurements.py`), producing the `val_`-prefixed outputs used by the validation stages.

**Physics inputs:** `csv/01_geometry.csv`, `stl/` and `validation_stl/` meshes, `utils/00_measured_mass_inertia.csv` (measured mass and trifilar oscillation time), `utils/00_validation_geometry.csv` (true geometry of the flight-tested props)

**Physics outputs:** `csv/03_naca_codes.csv`, `csv/03_mass_inertia.csv` (main corrected output for downstream use: `izz_regressed`), the calibration audit tables `csv/03_sls_density_validation.csv`, `csv/03_mass_inertia_validation_regression.csv`, `csv/03_mass_inertia_validation_error_summary.csv`, and the validation-subset outputs `csv/val_02_stl_volumes.csv`, `csv/val_03_mass_inertia.csv`, `csv/val_03_naca_codes.csv`

**Structure:**

1. Imports
2. Configuration — all tunable constants, paths and settings
3. Function definitions — STL loading, NACA code generation, mass/inertia extraction, regression helpers
4. Main code — measured-data loading and density calibration, NACA codes, mass/inertia over all configs, inertia correction, validation-subset processing and final checks


## 1. Imports

In [ ]:
import os
import sys

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
sys.dont_write_bytecode = True

from pathlib import Path

import numpy as np
import pandas as pd
import trimesh
from scipy.interpolate import CubicSpline
from scipy.spatial import ConvexHull
from tqdm.auto import tqdm

BASE_DIR = Path(globals().get('__vsc_ipynb_file__', Path.cwd())).parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
import pipeline_config as cfg
from utils import measurements

## 2. Configuration

In [ ]:
CSV_DIR = BASE_DIR / 'csv'
STL_DIR = BASE_DIR / 'stl'
VALIDATION_STL_DIR = measurements.validation_stl_dir(BASE_DIR)
GEOMETRY_CSV = CSV_DIR / cfg.CSV_NAMES['geometry']
STL_VOLUMES_CSV = CSV_DIR / cfg.CSV_NAMES['stl_volumes']
NACA_CSV = CSV_DIR / cfg.CSV_NAMES['naca_codes']
MASS_INERTIA_CSV = CSV_DIR / cfg.CSV_NAMES['mass_inertia']

STL_SEARCH_DIRS = [
    VALIDATION_STL_DIR,
    STL_DIR,
    BASE_DIR,
]

INNER_STATION_RADIUS_MM = cfg.INNER_ANCHOR_RADIUS_MM
HUB_OUTER_RADIUS_MM = cfg.HUB_STATION_RADIUS_MM
QPROP_STATION_FRACTIONS = cfg.QPROP_STATION_FRACTIONS
QPROP_STATION_NAMES = cfg.QPROP_STATION_NAMES

STL_FILENAME_PATTERNS = cfg.STL_FILENAME_PATTERNS

print('STL search dirs:')
for search_dir in STL_SEARCH_DIRS:
    print(f'  {search_dir}  exists={Path(search_dir).exists()}')

## 3. Function Definitions

- **load_raw_mesh(mesh_path)** — opens an STL file and returns its triangle mesh together with the watertight flag, ready for geometric analysis.
- **stl_match_sort_key(path)** — sort key used to pick the most specific match when an STL is located by glob (shortest filename first).
- **find_stl_path(config_id, search_dirs)** — locates the STL file for a config_id by trying the known folders and filename patterns, falling back to a recursive glob. Returns the path or None. The search order (`validation_stl/` before `stl/`) is intentional: the bench-measured props are matched to their actually printed meshes.
- **fit_naca_spline(r_inner, r_outer, val_inner, val_outer)** — natural cubic spline of a single NACA airfoil parameter between the inner and outer reference cross-sections. Returns the spline object.
- **naca_code_from_params(camber, camber_pos, thickness)** — formats camber, max-camber position and thickness into a standard NACA 4-digit code string; symmetric sections get camber position 0.
- **build_naca_row(row)** — for one propeller, splines the airfoil parameters along the radius, evaluates them at the hub, the six QProp stations and the three anchor stations, and returns the dictionary of NACA codes.
- **stl_mass_inertia(config_id, density_g_per_mm3, stl_dir)** — reads a propeller's STL, multiplies its volume by the effective SLS print density, and returns mass, centre of mass, the full inertia tensor about the origin and about the centre of mass, the spin-axis inertia `izz`, the radius of gyration, and the convex-hull frontal area. Returns a NaN record when the STL is missing or unreadable.
- **fit_linear_model(X, y)** — ordinary least-squares fit with intercept; returns the coefficients, intercept, predictions and R².
- **loo_cv_abs_error_pct(X, y)** — leave-one-out cross-validation of the same linear fit; returns the mean absolute percent error, showing whether the correction generalises rather than overfits.
- **chk(label, cond)** — pass/fail assertion helper used by the final checks.

The measured-data access and the trifilar-pendulum conversion live in `utils/measurements.py` and are shared with NB9: `load_measured_mass_inertia_by_id` (one averaged row per measured prop) and `add_measured_izz` (oscillation time to measured spin-axis inertia).


In [ ]:
def load_raw_mesh(mesh_path):

    mesh = trimesh.load_mesh(mesh_path)
    if isinstance(mesh, trimesh.Scene):
        mesh = trimesh.util.concatenate(tuple(mesh.dump()))

    return mesh, mesh.is_watertight


def stl_match_sort_key(path):

    return (len(path.name), str(path))


def find_stl_path(config_id, search_dirs=None):

    if search_dirs is None:
        search_dirs = STL_SEARCH_DIRS
    config_id = int(config_id)
    for search_dir in search_dirs:
        search_dir = Path(search_dir)
        if not search_dir.exists():
            continue
        for pattern in STL_FILENAME_PATTERNS:
            candidate = search_dir / pattern.format(config_id=config_id)
            if candidate.exists():

                return candidate
    for search_dir in search_dirs:
        search_dir = Path(search_dir)
        if not search_dir.exists():
            continue
        matches = list(search_dir.glob(f'**/*{config_id}*.stl'))
        if matches:
            matches.sort(key=stl_match_sort_key)

            return matches[0]

    return None

In [ ]:
def fit_naca_spline(r_inner, r_outer, val_inner, val_outer):

    r = np.array([r_inner, r_outer], dtype=float)
    values = np.array([val_inner, val_outer], dtype=float)

    return CubicSpline(r, values, bc_type='natural')


def naca_code_from_params(camber, camber_pos, thickness):

    c = int(np.clip(round(float(camber)), 0, 9))
    p = int(np.clip(round(float(camber_pos)), 0, 9))
    t = int(np.clip(round(float(thickness)), 1, 99))
    if c == 0:
        p = 0

    return f'{c}{p}{t:02d}'


def build_naca_row(row):

    r_inner = float(INNER_STATION_RADIUS_MM)
    r_outer = float(row['radius_mm'])
    r_mid = float(row['mid_radial_pos']) * r_outer
    r_hub = float(HUB_OUTER_RADIUS_MM)

    inner_camber = float(row['inner_camber'])
    inner_camber_pos = float(row['inner_max_pos'])
    inner_thickness = float(row['inner_thickness_pct'])
    outer_camber = float(row['outer_camber'])
    outer_camber_pos = float(row['outer_max_pos'])
    outer_thickness = float(row['outer_thickness_pct'])

    spline_camber = fit_naca_spline(r_inner, r_outer, inner_camber, outer_camber)
    spline_camber_pos = fit_naca_spline(r_inner, r_outer, inner_camber_pos, outer_camber_pos)
    spline_thickness = fit_naca_spline(r_inner, r_outer, inner_thickness, outer_thickness)

    def naca_at(r_mm):

        return naca_code_from_params(spline_camber(r_mm), spline_camber_pos(r_mm), spline_thickness(r_mm))

    naca_hub = naca_at(r_hub)
    naca_stns = {}
    for name, fraction in zip(QPROP_STATION_NAMES, QPROP_STATION_FRACTIONS):
        r_mm = fraction * r_outer
        if r_mm <= r_hub:
            r_mm = r_hub + 0.1
        naca_stns[name] = naca_at(r_mm)

    naca_inner = naca_code_from_params(inner_camber, inner_camber_pos, inner_thickness)
    naca_mid = naca_at(r_mid)
    naca_outer = naca_code_from_params(outer_camber, outer_camber_pos, outer_thickness)

    result = {'config_id': int(row['config_id'])}
    result['naca_inner'] = naca_inner
    result['naca_hub'] = naca_hub
    for name in QPROP_STATION_NAMES:
        result[f'naca_{name}'] = naca_stns[name]
    result['naca_outer'] = naca_outer
    result['naca_mid'] = naca_mid

    return result

In [ ]:
def stl_mass_inertia(config_id, density_g_per_mm3, stl_dir=None):

    if stl_dir is None:
        stl_path = find_stl_path(config_id)
    else:
        stl_path = find_stl_path(config_id, search_dirs=[Path(stl_dir)])

    nan_record = {
        'config_id': int(config_id), 'stl_ok': False,
        'watertight': False,
        'stl_path': None,
        'vol_raw_mm3': np.nan,
        'mass_kg': np.nan, 'mass_g': np.nan,
        'com_x_mm': np.nan, 'com_y_mm': np.nan, 'com_z_mm': np.nan,
        'ixx_com': np.nan, 'iyy_com': np.nan, 'izz_com': np.nan,
        'ixx': np.nan, 'iyy': np.nan, 'izz': np.nan,
        'ixy': np.nan, 'ixz': np.nan, 'iyz': np.nan,
        'izz_com_correction_kg_m2': np.nan,
        'radius_gyration_m': np.nan,
        'frontal_area_m2': np.nan,
    }

    if stl_path is None:

        return nan_record

    try:
        mesh, wt = load_raw_mesh(str(stl_path))
        vol_raw = abs(float(mesh.volume))

        mass_kg = vol_raw * density_g_per_mm3 / 1000.0
        mass_g = mass_kg * 1000.0

        density_kg_per_mm3 = density_g_per_mm3 / 1000.0
        mesh.density = density_kg_per_mm3

        I_com_mm = np.asarray(mesh.moment_inertia, dtype=float)
        com_mm = np.asarray(mesh.center_mass, dtype=float)

        r = com_mm.reshape(3)
        I_parallel_mm = mass_kg * ((np.dot(r, r) * np.eye(3)) - np.outer(r, r))
        I_origin_mm = I_com_mm + I_parallel_mm

        I_com_m = I_com_mm * 1e-6
        I_origin_m = I_origin_mm * 1e-6
        izz_com_correction = I_parallel_mm[2, 2] * 1e-6
        izz_origin = float(I_origin_m[2, 2])
        if mass_kg > 0 and izz_origin > 0:
            radius_gyration_m = np.sqrt(izz_origin / mass_kg)
        else:
            radius_gyration_m = np.nan

        xy = mesh.vertices[:, :2]
        hull_2d = ConvexHull(xy)
        frontal_area_m2 = float(hull_2d.volume) * 1e-6

        return {
            'config_id':       int(config_id),
            'stl_ok':          True,
            'watertight':      bool(wt),
            'stl_path':        str(stl_path),
            'vol_raw_mm3':     round(vol_raw, 4),
            'mass_kg':         round(mass_kg, 8),
            'mass_g':          round(mass_g, 5),
            'com_x_mm':        round(float(com_mm[0]), 6),
            'com_y_mm':        round(float(com_mm[1]), 6),
            'com_z_mm':        round(float(com_mm[2]), 6),
            'ixx_com':         round(float(I_com_m[0, 0]), 10),
            'iyy_com':         round(float(I_com_m[1, 1]), 10),
            'izz_com':         round(float(I_com_m[2, 2]), 10),
            'ixx':             round(float(I_origin_m[0, 0]), 10),
            'iyy':             round(float(I_origin_m[1, 1]), 10),
            'izz':             round(izz_origin, 10),
            'ixy':             round(float(I_origin_m[0, 1]), 10),
            'ixz':             round(float(I_origin_m[0, 2]), 10),
            'iyz':             round(float(I_origin_m[1, 2]), 10),
            'izz_com_correction_kg_m2': round(float(izz_com_correction), 10),
            'radius_gyration_m': round(float(radius_gyration_m), 10),
            'frontal_area_m2': round(frontal_area_m2, 8),
        }
    except Exception as e:
        print(f'  [ERROR] config {config_id}: {e}')

        return nan_record

In [ ]:
def fit_linear_model(X, y):

    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    X_design = np.column_stack([X, np.ones(len(X))])
    beta, residuals_unused, rank_unused, singular_values_unused = np.linalg.lstsq(X_design, y, rcond=None)
    coef = beta[:-1]
    intercept = float(beta[-1])
    pred = X @ coef + intercept
    ss_res = float(np.sum((y - pred) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    if ss_tot > 0:
        r2 = 1.0 - ss_res / ss_tot
    else:
        r2 = np.nan

    return coef, intercept, pred, r2


def loo_cv_abs_error_pct(X, y):

    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(y)
    if n < 3:

        return float('nan')

    Xd = np.column_stack([X, np.ones(n)])
    errs = []
    for i in range(n):
        mask = np.ones(n, dtype=bool)
        mask[i] = False
        beta, residuals_unused, rank_unused, singular_values_unused = np.linalg.lstsq(Xd[mask], y[mask], rcond=None)
        pred_i = float(Xd[i] @ beta)
        if pred_i > 0:
            errs.append(abs(y[i] - pred_i) / pred_i * 100.0)
    if errs:
        mean_abs_error = float(np.mean(errs))
    else:
        mean_abs_error = float('nan')

    return mean_abs_error


def chk(label, cond):

    global all_ok
    status = ' OK ' if cond else 'FAIL'
    print(f'  [{status}]  {label}')
    if not cond:
        all_ok = False

## 4. Main Code

The main code loads the geometry table and the measured bench data, calibrates the effective SLS print density from the measured props and their printed STLs, generates the per-station NACA codes, computes mass and inertia for all 5000 STLs, fits and applies the validation-based inertia correction, saves all outputs, re-runs the same mass/inertia and NACA computations for the 10 recovered validation props, and closes with pass/fail checks.


### 4.1 Load Geometry

In [ ]:
GEO_COLS = [
    'config_id', 'radius_mm', 'ring_height_mm', 'ring_thickness_mm', 'blade_count',
    'inner_thickness_pct', 'inner_max_pos', 'inner_camber', 'inner_chord_mm', 'inner_angle_deg',
    'mid_radial_pos', 'mid_chord_mm', 'mid_angle_deg',
    'outer_thickness_pct', 'outer_max_pos', 'outer_camber', 'outer_chord_mm', 'outer_angle_deg',
]

if not GEOMETRY_CSV.exists():
    raise FileNotFoundError(f'Geometry file not found: {GEOMETRY_CSV}. Run notebook 1 first.')

geo = pd.read_csv(GEOMETRY_CSV)
missing = []
for column in GEO_COLS:
    if column not in geo.columns:
        missing.append(column)
if missing:
    raise ValueError(f'Missing columns in geometry CSV: {missing}')

geo = geo[GEO_COLS].sort_values('config_id').reset_index(drop=True)
print(f'Loaded {len(geo)} configurations from {GEOMETRY_CSV}')

### 4.2 Measured Data and SLS Density Calibration

Loads the bench measurements through `utils/measurements.py` (one averaged row per measured prop) and estimates the effective SLS print density from the measured masses and their printed STL volumes. The density used by the 5000-prop virtual pipeline is the weighted effective density

$\rho_\mathrm{SLS} = \frac{\sum m_\mathrm{measured}}{\sum V_\mathrm{STL}}$


In [ ]:
meas_validation = measurements.load_measured_mass_inertia(BASE_DIR)
meas_validation_by_id = measurements.load_measured_mass_inertia_by_id(BASE_DIR)

print(f'Loaded measured CSV       : {measurements.measured_mass_inertia_path(BASE_DIR)}')
print(f'Valid measured rows       : {len(meas_validation)}')
print(f'Unique valid config IDs   : {len(meas_validation_by_id)}')

CALIBRATION_CONFIGS = list(meas_validation_by_id[['config_id', 'mass_g']].itertuples(index=False, name=None))

density_records = []
missing_density_stls = []

print(f"  {'config':>8}  {'mass g':>7}  {'vol_raw mm³':>12}  {'watertight':>10}  {'density g/cm³':>14}  {'stl file':<30}")
print(f"  {'-' * 100}")

for config_id, mass_g in CALIBRATION_CONFIGS:
    stl_path = find_stl_path(config_id)
    if stl_path is None:
        missing_density_stls.append(int(config_id))
        print(f'  [WARN] STL not found for config {config_id} in STL_SEARCH_DIRS')
        continue
    mesh, wt = load_raw_mesh(str(stl_path))
    vol_raw = abs(float(mesh.volume))
    if not np.isfinite(vol_raw) or vol_raw <= 0:
        print(f'  [WARN] Invalid STL volume for config {config_id}: {vol_raw}')
        continue
    density_g_per_mm3 = mass_g / vol_raw
    density_records.append({
        'config_id': int(config_id),
        'mass_g': float(mass_g),
        'vol_raw_mm3': vol_raw,
        'density_g_per_mm3': density_g_per_mm3,
        'density_g_per_cm3': density_g_per_mm3 * 1000.0,
        'watertight': bool(wt),
        'stl_path': str(stl_path),
    })
    print(f"  {int(config_id):>8}  {mass_g:>7.2f}  {vol_raw:>12.2f}  {str(bool(wt)):>10}  {density_g_per_mm3 * 1000:>14.4f}  {stl_path.name:<30}")

density_validation_df = pd.DataFrame(density_records)

if len(density_validation_df):
    SLS_DENSITY_G_PER_MM3 = float(density_validation_df['mass_g'].sum() / density_validation_df['vol_raw_mm3'].sum())
    SLS_DENSITY_G_PER_CM3 = SLS_DENSITY_G_PER_MM3 * 1000.0
    PLA_DENSITY_G_PER_MM3 = SLS_DENSITY_G_PER_MM3
    PLA_DENSITY_G_PER_CM3 = SLS_DENSITY_G_PER_CM3
    print()
    print('SLS effective density from validation set')
    print(f'  Weighted density : {SLS_DENSITY_G_PER_CM3:.4f} g/cm³')
    print(f'  Mean individual  : {density_validation_df["density_g_per_cm3"].mean():.4f} g/cm³')
    print(f'  Std individual   : {density_validation_df["density_g_per_cm3"].std():.4f} g/cm³')
    print(f'  n matched STLs   : {len(density_validation_df)} / {len(CALIBRATION_CONFIGS)}')
    if missing_density_stls:
        print(f'  missing STLs     : {len(missing_density_stls)}')
        print(missing_density_stls)
else:
    SLS_DENSITY_G_PER_CM3 = 1.00
    SLS_DENSITY_G_PER_MM3 = SLS_DENSITY_G_PER_CM3 / 1000.0
    PLA_DENSITY_G_PER_CM3 = SLS_DENSITY_G_PER_CM3
    PLA_DENSITY_G_PER_MM3 = SLS_DENSITY_G_PER_MM3
    print(f'[WARN] No validation density data — using fallback density: {SLS_DENSITY_G_PER_CM3} g/cm³')

### 4.3 NACA Code Generation

Evaluates the airfoil-parameter splines at the hub, the six QProp stations and the three anchor stations for every propeller, and saves the zero-padded 4-digit codes to `csv/03_naca_codes.csv`.


In [ ]:
naca_records = []
for row_index, row in geo.iterrows():
    naca_records.append(build_naca_row(row))
naca_df = pd.DataFrame(naca_records).sort_values('config_id').reset_index(drop=True)

naca_cols = []
for column in naca_df.columns:
    if column.startswith('naca_'):
        naca_cols.append(column)
for column in naca_cols:
    naca_df[column] = naca_df[column].astype('string').str.zfill(4)

CSV_DIR.mkdir(parents=True, exist_ok=True)
naca_df.to_csv(NACA_CSV, index=False)

station_list = []
for name in QPROP_STATION_NAMES:
    station_list.append(f'naca_{name}')
station_names_text = ', '.join(station_list)

print(f'Saved {len(naca_df)} rows → {NACA_CSV}')
print(f'Columns: {list(naca_df.columns)}')
print()
print('Column order   : config_id, naca_inner, naca_hub, ' + station_names_text + ', naca_outer, naca_mid')
print('QProp stations : naca_hub, ' + station_names_text)
print('Traceability   : naca_inner, naca_outer, naca_mid')
print()
naca_df.head()

### 4.4 Mass and Inertia From STL

Applies `stl_mass_inertia` to every propeller at the calibrated density, then summarises the raw STL volumes.


In [ ]:
records = []
for config_id in tqdm(geo['config_id'].astype(int), desc='STL mass/inertia'):
    records.append(stl_mass_inertia(config_id, PLA_DENSITY_G_PER_MM3))

mass_inertia_df = pd.DataFrame(records).sort_values('config_id').reset_index(drop=True)
stl_ok_count = mass_inertia_df['stl_ok'].sum()
wt_count = mass_inertia_df['watertight'].sum()

print(f'Successful  : {stl_ok_count} / {len(mass_inertia_df)}')
print(f'Watertight  : {wt_count} / {stl_ok_count}')
print(f'Not watertight: {stl_ok_count - wt_count} / {stl_ok_count}')

ok = mass_inertia_df[mass_inertia_df['stl_ok']]
print()
print('vol_raw_mm3 stats:')
print(ok['vol_raw_mm3'].describe().round(2).to_string())
print()
print(f'Watertight raw STLs : {ok["watertight"].sum()} / {len(ok)}')

### 4.5 Representative-Based Inertia Correction and Save Outputs

Converts the measured trifilar periods into measured spin-axis inertias (`utils/measurements.py`), matches them to the STL-derived inertias, fits the single 2-parameter correction `Izz_measured = a · Izz_STL + b`, cross-checks it with leave-one-out cross-validation, applies it to the full 5000-prop table, and saves all outputs.


In [ ]:
validation_stl_records = []
for config_id in tqdm(meas_validation_by_id['config_id'].astype(int), desc='Validation STL mass/inertia'):
    validation_stl_records.append(stl_mass_inertia(config_id, PLA_DENSITY_G_PER_MM3))

validation_stl_df = pd.DataFrame(validation_stl_records)

meas_izz_df = measurements.add_measured_izz(meas_validation_by_id)

validation_compare = meas_izz_df.merge(validation_stl_df, on='config_id', how='inner', suffixes=('_meas', '_sim'))
validation_compare = validation_compare[validation_compare['stl_ok'] & np.isfinite(validation_compare['izz']) & np.isfinite(validation_compare['izz_meas_kg_m2']) & (validation_compare['izz'] > 0) & (validation_compare['izz_meas_kg_m2'] > 0)].copy()

if len(validation_compare) < 2:
    raise ValueError('Not enough valid validation props to fit Izz correction. Check STL_SEARCH_DIRS and STL_FILENAME_PATTERNS.')

validation_compare['izz_error_pct_raw'] = (validation_compare['izz_meas_kg_m2'] - validation_compare['izz']) / validation_compare['izz'] * 100.0

X_simple = validation_compare[['izz']].to_numpy(dtype=float)
y = validation_compare['izz_meas_kg_m2'].to_numpy(dtype=float)
coef_simple, intercept_simple, pred_simple, r2_simple = fit_linear_model(X_simple, y)
validation_compare['izz_regressed_kg_m2'] = pred_simple
validation_compare['izz_error_pct_after_regression'] = (validation_compare['izz_meas_kg_m2'] - validation_compare['izz_regressed_kg_m2']) / validation_compare['izz_regressed_kg_m2'] * 100.0

raw_abs_err = validation_compare['izz_error_pct_raw'].abs().mean()
ins_abs_err = validation_compare['izz_error_pct_after_regression'].abs().mean()
loo_abs_err = loo_cv_abs_error_pct(X_simple, y)

print('Validation-based Izz correction  (single 2-parameter model)')
print(f'  matched validation props : {len(validation_compare)}')
print(f'  model                    : Izz_measured = {coef_simple[0]:.6g} * Izz_STL + {intercept_simple:.6g}')
print(f'  in-sample R2             : {r2_simple:.4f}')
print()
print('  Mean ABS error vs measured Izz [%]:')
print(f'    raw STL (no correction)     : {raw_abs_err:.2f}%')
print(f'    after correction (in-sample): {ins_abs_err:.2f}%')
print(f'    after correction (LOO-CV)   : {loo_abs_err:.2f}%   <- out-of-sample; ~in-sample => no over-fit')
print()

mass_inertia_df['izz_regression_slope'] = float(coef_simple[0])
mass_inertia_df['izz_regression_intercept'] = float(intercept_simple)
mass_inertia_df['izz_regressed'] = coef_simple[0] * mass_inertia_df['izz'] + intercept_simple
mass_inertia_df.loc[~mass_inertia_df['stl_ok'], 'izz_regressed'] = np.nan
mass_inertia_df.loc[mass_inertia_df['izz_regressed'] <= 0, 'izz_regressed'] = np.nan

validation_error_summary = pd.DataFrame({
    'stage': ['raw_stl_com_corrected', 'linear_izz_regression', 'linear_izz_regression_loo_cv'],
    'mean_error_pct': [validation_compare['izz_error_pct_raw'].mean(), validation_compare['izz_error_pct_after_regression'].mean(), np.nan],
    'mean_abs_error_pct': [raw_abs_err, ins_abs_err, loo_abs_err],
    'std_error_pct': [validation_compare['izz_error_pct_raw'].std(), validation_compare['izz_error_pct_after_regression'].std(), np.nan],
})

display(validation_error_summary)

OUTPUT_COLS = [
    'config_id', 'stl_ok', 'watertight', 'stl_path',
    'vol_raw_mm3',
    'mass_kg', 'mass_g',
    'com_x_mm', 'com_y_mm', 'com_z_mm',
    'ixx_com', 'iyy_com', 'izz_com',
    'ixx', 'iyy', 'izz', 'ixy', 'ixz', 'iyz',
    'izz_com_correction_kg_m2',
    'radius_gyration_m',
    'izz_regressed', 'izz_regression_slope', 'izz_regression_intercept',
    'frontal_area_m2',
]

mass_inertia_df[OUTPUT_COLS].to_csv(MASS_INERTIA_CSV, index=False)
validation_compare.to_csv(CSV_DIR / '03_mass_inertia_validation_regression.csv', index=False)
density_validation_df.to_csv(CSV_DIR / '03_sls_density_validation.csv', index=False)
validation_error_summary.to_csv(CSV_DIR / '03_mass_inertia_validation_error_summary.csv', index=False)

print(f'Saved {len(mass_inertia_df)} rows -> {MASS_INERTIA_CSV}')
print(f'Saved validation regression table -> {CSV_DIR / "03_mass_inertia_validation_regression.csv"}')
print(f'Saved validation error summary -> {CSV_DIR / "03_mass_inertia_validation_error_summary.csv"}')
print(f'Saved SLS density validation table -> {CSV_DIR / "03_sls_density_validation.csv"}')

### 4.6 Representative Props — Mass and Inertia Check

Prints the measured-vs-predicted comparison for every measured propeller.


In [ ]:
print('Validation props — measured vs STL/regressed inertia')
print(f'Rig: r = {measurements.PENDULUM_RADIUS_M * 1000:.0f} mm · L_string = {measurements.PENDULUM_STRING_M * 1000:.0f} mm · T_tare_10osc = {measurements.PLATE_TARE_TIME_S:.3f} s')
print(f'Correction  : Izz_measured = {coef_simple[0]:.6g} * Izz_STL + {intercept_simple:.6g}')
print()

cols_to_show = [
    'config_id',
    'mass_g_meas', 'mass_g_sim',
    'T_meas', 'T_period_s',
    'izz_meas_kg_m2', 'izz', 'radius_gyration_m',
    'izz_regressed_kg_m2',
    'izz_error_pct_raw',
    'izz_error_pct_after_regression',
    'n_measurements',
]

display_df = validation_compare.copy()
if 'mass_g_meas' not in display_df.columns and 'mass_g_x' in display_df.columns:
    display_df = display_df.rename(columns={'mass_g_x': 'mass_g_meas'})
if 'mass_g_sim' not in display_df.columns and 'mass_g_y' in display_df.columns:
    display_df = display_df.rename(columns={'mass_g_y': 'mass_g_sim'})

existing_cols = []
for column in cols_to_show:
    if column in display_df.columns:
        existing_cols.append(column)

display(display_df[existing_cols].sort_values('config_id').reset_index(drop=True))

### 4.7 Validation Subset — Recovered Geometry

The 10 early-printed validation propellers keep their real config_ids, but a later re-run of NB1 reassigned those ids to different geometries, so their rows in `csv/01_geometry.csv` no longer describe the printed parts. Their true geometry is stored in `utils/00_validation_geometry.csv` (rows with `geometry_source = 'validation_sample'`) and their printed meshes in `validation_stl/`. This section runs the identical mass/inertia and NACA computations on those 10 props — reusing the SLS density and the inertia regression fitted above, since both are global material/rig calibrations — and writes the `val_`-prefixed outputs that the validation stages of NB4–NB6b consume.


In [ ]:
val_geo = measurements.load_recovered_validation_geometry(BASE_DIR)
print(f'Recovered validation props: {len(val_geo)} -> {sorted(val_geo["config_id"].tolist())}')

val_records = []
for config_id in val_geo['config_id']:
    val_records.append(stl_mass_inertia(int(config_id), PLA_DENSITY_G_PER_MM3, stl_dir=VALIDATION_STL_DIR))
val_mass_df = pd.DataFrame(val_records)

val_mass_df['izz_regressed'] = float(coef_simple[0]) * val_mass_df['izz'] + float(intercept_simple)
val_mass_df['izz_regression_slope'] = float(coef_simple[0])
val_mass_df['izz_regression_intercept'] = float(intercept_simple)

val_mass_df = val_mass_df.sort_values('config_id').reset_index(drop=True)
val_mass_df.to_csv(CSV_DIR / 'val_03_mass_inertia.csv', index=False)
print(f'Saved {len(val_mass_df)} rows -> val_03_mass_inertia.csv')
print(val_mass_df[['config_id', 'stl_ok', 'vol_raw_mm3', 'mass_g', 'izz', 'izz_regressed', 'frontal_area_m2']].to_string(index=False))

val_stl_volumes_df = pd.DataFrame({
    'config_id': val_mass_df['config_id'],
    'volume_L': val_mass_df['vol_raw_mm3'] / 1e6,
    'volume_m3': val_mass_df['vol_raw_mm3'] * 1e-9,
    'stl_ok': val_mass_df['stl_ok'],
    'single_solid': val_mass_df['stl_ok'],
})
val_stl_volumes_df.to_csv(CSV_DIR / 'val_02_stl_volumes.csv', index=False)
print(f'Saved {len(val_stl_volumes_df)} rows -> val_02_stl_volumes.csv')

val_naca_records = []
for row_index, row in val_geo.iterrows():
    val_naca_records.append(build_naca_row(row))
val_naca_df = pd.DataFrame(val_naca_records).sort_values('config_id').reset_index(drop=True)
for column in val_naca_df.columns:
    if column.startswith('naca_'):
        val_naca_df[column] = val_naca_df[column].astype('string').str.zfill(4)
val_naca_df.to_csv(CSV_DIR / 'val_03_naca_codes.csv', index=False)
print(f'Saved {len(val_naca_df)} rows -> val_03_naca_codes.csv')
val_naca_df

### 4.8 Final Checks

In [ ]:
all_ok = True

chk('NACA row count matches geometry', len(naca_df) == len(geo))
chk('NACA config_id unique', naca_df['config_id'].is_unique)
chk('No NaN in NACA codes', naca_df[['naca_hub', 'naca_inner', 'naca_mid', 'naca_outer']].notna().all().all())

for column in ['naca_hub', 'naca_inner', 'naca_mid', 'naca_outer']:
    codes_valid = True
    for value in naca_df[column]:
        if len(str(value)) != 4 or not str(value).isdigit():
            codes_valid = False
    chk(f'{column} are valid 4-digit strings', codes_valid)

chk('Mass/inertia row count matches geometry', len(mass_inertia_df) == len(geo))
chk('mass_kg > 0 for all stl_ok configs', (mass_inertia_df.loc[mass_inertia_df['stl_ok'], 'mass_kg'] > 0).all())
chk('izz > 0 for all stl_ok configs', (mass_inertia_df.loc[mass_inertia_df['stl_ok'], 'izz'] > 0).all())
chk('radius_gyration_m > 0 for all stl_ok configs', (mass_inertia_df.loc[mass_inertia_df['stl_ok'], 'radius_gyration_m'] > 0).all())
chk('izz_regressed > 0 for all stl_ok configs', (mass_inertia_df.loc[mass_inertia_df['stl_ok'], 'izz_regressed'] > 0).all())
chk('Validation regression has at least 50 props', len(validation_compare) >= 50)
chk('NACA CSV exists', NACA_CSV.exists())
chk('Mass/inertia CSV exists', MASS_INERTIA_CSV.exists())
chk('Validation regression CSV exists', (CSV_DIR / '03_mass_inertia_validation_regression.csv').exists())
chk('Validation error summary CSV exists', (CSV_DIR / '03_mass_inertia_validation_error_summary.csv').exists())
chk('SLS density validation CSV exists', (CSV_DIR / '03_sls_density_validation.csv').exists())
chk('Validation subset rows == recovered geometry rows', len(val_mass_df) == len(val_geo))
chk('Validation subset STLs all found', bool(val_mass_df['stl_ok'].all()))
chk('Validation subset NACA CSV exists', (CSV_DIR / 'val_03_naca_codes.csv').exists())

print()
print('=' * 60)
print('ALL CHECKS PASSED' if all_ok else 'SOME CHECKS FAILED')
print('=' * 60)